In [ ]:
%load_ext watermark


In [ ]:
import logging

from IPython.display import display, HTML
from matplotlib import pyplot as plt
import pandas as pd
from retry import retry
from slugify import slugify
from teeplot import teeplot as tp

from pylib._percentilestatcat_plot import (
    percentilestatcat_plot,
)
from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-17-vanilla-compscreen"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
df = retry(tries=5, logger=logging.getLogger(__name__))(pd.read_parquet)(
    "https://osf.io/3ac7f/download",
)

df.head()


## Plot Data


In [ ]:
for trt_name, group in df.groupby("trt_name", observed=True):
    display(HTML(f"<h1>{trt_name}</h1>"))

    dfx = group[
        (group["replicate_uuid"] == group["replicate_uuid"].unique()[0])
    ].reset_index(drop=True)

    dfx_ = dfx.copy()
    dfx_["is_focal_defmut"] = "null"
    data = pd.concat([dfx, dfx_], ignore_index=True)

    for y in (
        "defmut_norm_all-num_leaves",
        # "defmut_norm_ot_delta:4-num_leaves",
        # "defmut_norm_ot_delta:7-num_leaves",
        "defmut_norm_ot_delta:14-num_leaves",
        # "defmut_norm_ot_delta:28-num_leaves",
        # "defmut_norm_ot_delta:44-num_leaves",
        "defmut_norm_match:variant_flavor-num_leaves",
        "defmut_norm_all-clade_duration",
        # "defmut_norm_ot_delta:4-clade_duration",
        # "defmut_norm_ot_delta:7-clade_duration",
        "defmut_norm_ot_delta:14-clade_duration",
        # "defmut_norm_ot_delta:28-clade_duration",
        # "defmut_norm_ot_delta:44-clade_duration",
        "defmut_norm_match:variant_flavor-clade_duration",
    ):
        display(HTML(f"<h2>{trt_name} {y}</h2>"))
        plt.close("all")  # release resources
        with tp.teed(
            percentilestatcat_plot,
            data=data,
            x="is_focal_defmut",
            y=y,
            hue="is_focal_defmut",
            teeplot_outattrs={
                "trt_name": slugify(trt_name),
            },
            teeplot_subdir=teeplot_subdir,
        ):
            pass
